#### Libraries

In [ ]:
import os, sys
env = r"C:\Users\Costanza\anaconda3\envs\Dati_climatici"
os.environ["ECCODES_DIR"] = env + r"\Library"
os.add_dll_directory(env + r"\Library\bin")
os.environ["PATH"] = env + r"\Library\bin;" + os.environ["PATH"]
import eccodes
import earthkit.data
import json
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

#### Functions

In [ ]:
def download_cds_dataset(dataset, request):
    """
    Download a CDS dataset and return it as an xarray Dataset.
    This wrapper works for both CORDEX-style and ERA5-Land-style requests.
    """
    ds = earthkit.data.from_source("cds", dataset, request)
    return ds.to_xarray(xarray_open_mfdataset_kwargs=dict(decode_cf=True, decode_times=True))

def download_dfs(requests, dictionay_name, dataset="projections-cordex-domains-single-levels"):
    """
    Function to "download" as a temporary file with Earthkit and add the orography/variables of several climate models to the dictionary (dfs_orography or dfs_models).
    :param requests: the parameters of the request (those indicated in the Copernicus dataset: names RCM, GCM, resolution, etc.).
    Example of a request
    irnet_orography = {
    "domain": "europe",
    "experiment": "historical",
    "horizontal_resolution": "0_11_degree_x_0_11_degree",
    "temporal_resolution": "fixed",
    "variable": ["orography"],
    "gcm_model": "ichec_ec_earth",
    "rcm_model": "knmi_racmo22e",
    "ensemble_member": "r1i1p1"
    }
    :param dictionary_name: the name of the dictionary to save the dataset in.
    :param dataset: CDS dataset identifier.
    """
    for key_name, request in requests:
        try:
            ds = download_cds_dataset(dataset, request)
            dictionay_name[key_name] = ds
            print(f"Dataset '{key_name}' downloaded and added to dictionary")

            # Convert tasmax from Kelvin to Celsius
            if "tasmax" in ds:
                ds["tasmax"] = ds["tasmax"] - 273.15
                ds["tasmax"].attrs["units"] = "°C"
            # Convert tasmin from Kelvin to Celsius
            if "tasmin" in ds:
                ds["tasmin"] = ds["tasmin"] - 273.15
                ds["tasmin"].attrs["units"] = "°C"
            # Convert pr from kg*m-2*s-1 to mm
            if "pr" in ds:
                ds["pr"] = ds["pr"] * 100000
                ds["pr"].attrs["units"] = "mm"
            # Convertion not needed for wind speed (already in m*s-1)

        except Exception as e:
            print(f"Error with the dataset '{key_name}': {e}")

def run_all_models(file_json, variable):
    """
    Load climate model configurations from a JSON file, download the corresponding
    datasets for each model and time interval, and return all results in a structured
    dictionary.

    This function reads a JSON file containing multiple climate model specifications
    (GCM, RCM, ensemble member, and temporal ranges). For each model, it iterates
    through the defined start/end year pairs, builds the request parameters, and
    downloads the associated datasets using `download_dfs()`. All downloaded data
    are organized into a nested dictionary structure.

    Parameters:
        file_json (str):
            Path or filename of the JSON file containing model configurations (e.g., "tasmax.json").
        variable (str):
            Name of the climate variable to download (e.g., "maximum_2m_temperature_in_the_last_24_hours"
                                                            "minimum_2m_temperature_in_the_last_24_hours"
                                                            "mean_precipitation_flux"
                                                            "10m_wind_speed").
    Returns:
        dict:
            A dictionary where each key corresponds to a model name, and each value
            contains another dictionary mapping each temporal chunk (start_year–end_year)
            to the downloaded dataset.
    """

    dfs_models = {}                                                  # Main dictionary
    base_dir = os.getcwd()                                           # base_dir = os.getcwd() obtain current folder
    with open(os.path.join(base_dir, file_json)) as f:
        models = json.load(f)                                        # Read models from json file
    for model in models:                                             # Iterate through the models and download the data
        gcm = models[model]["gcm"]
        rcm = models[model]["rcm"]
        # ensemble_orography = models[model]["ensemble_orography"]
        ensemble_model = models[model]["ensemble_model"]
        start_years = models[model]["start_years"]
        end_years = models[model]["end_years"]
        dfs_chunks = {}                                              # Dictionary to store temporal chunks. Chunks can be: 1950-1950,1951-1955,...,1971-1975

        for sy, ey in zip(start_years, end_years):                   # Loop over each temporal chunk

            request_models = [
                (
                    f"{model}_{sy}_{ey}",
                    {
                        "domain": "europe",                                         # fixed
                        "experiment": ["historical", "rcp_4_5"],                    # change with rcp_8_5
                        "horizontal_resolution": "0_11_degree_x_0_11_degree",       # fixed
                        "temporal_resolution": "daily_mean",                        # fixed
                        "variable": variable,                                       # Argument of the function
                        "gcm_model": gcm,                                           # fixed
                        "rcm_model": rcm,                                           # fixed
                        "ensemble_member": ensemble_model,                          # fixed
                        "start_year": sy,                                           # Single start year
                        "end_year": ey,                                             # Single end year
                    },
                ),
            ]

            download_dfs(request_models, dfs_chunks)                  # Download of the dictionary
        dfs_models[model] = dfs_chunks                                # Fill the main dictionary
    return dfs_models





    """
    Save all the data in a .csv file, and export it.
    """

    all_models = []

    for model, windows in dfs.items():
        print("Processing model:", model)

        df_model = None

        for (start, end), da in windows.items():
            da = da.compute()

            lat2d  = da["lat"].values
            lon2d  = da["lon"].values
            rlat1d = da["rlat"].values
            rlon1d = da["rlon"].values
            vals   = da.values

            nlat, nlon = vals.shape
            records = []

            for i in range(nlat):
                for j in range(nlon):
                    v = vals[i, j]
                    if np.isnan(v):
                        continue

                    records.append({
                        "lat":  float(lat2d[i, j]),
                        "lon":  float(lon2d[i, j]),
                        "rlat": float(rlat1d[i]),
                        "rlon": float(rlon1d[j]),
                        f"{label}_{start}_{end}": float(v)
                    })

            df_window = pd.DataFrame.from_records(records)

            if df_model is None:
                df_model = df_window
            else:
                df_model = df_model.merge(
                    df_window,
                    on=["lat","lon","rlat","rlon"],
                    how="outer"
                )

        df_model.insert(0, "model", model)

        all_models.append(df_model)

    final_df = pd.concat(all_models, ignore_index=True)
    final_df.to_csv(
        out_filename,
        index=False,
        sep=";",
        decimal=",",
        float_format="%.6f"
    )

    print("CSV saved:", len(final_df), "rows ->", out_filename)

#### Download CERRA
CERRA sub-daily regional reanalysis data for Europe on single levels from 1984 to present
(https://cds.climate.copernicus.eu/datasets/reanalysis-cerra-single-levels?tab=overview)

In [ ]:
# Anni da scaricare
years = range(1994, 2007)

# Mesi
months = ["01", "02", "03", "04", "05", "10", "11", "12"]


# Ore CERRA: analisi ogni 3 ore
times = [
    "00:00"
]

# Cartella output
output_dir = Path(r"D:\DOTTORATO\16_GITHUB\HBM-ABCSubSim\CERRA sub-daily reanalysis copernicus\CERRA data")
output_dir.mkdir(parents=True, exist_ok=True)

for year in years:

    output_file = output_dir / f"SDWE_cerra_{year}.nc"

    if output_file.exists():
        print(f"{output_file.name} already exists, skip.")
        continue

    print(f"\n===== Download year {year} =====")

    datasets = []

    for month in months:

        print(f"Downloading {year}-{month}...")

        request_depth = {
            "variable": ["snow_depth_water_equivalent"],
            "level_type": "surface_or_atmosphere",
            "data_type": ["reanalysis"],
            "product_type": "analysis",

            "year": [str(year)],
            "month": [month],

            "day": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10",
                    "11", "12", "13", "14", "15", "16", "17", "18", "19", "20",
                    "21", "22", "23", "24", "25", "26", "27", "28", "29", "30", "31"],
                # f"{d:02d}" for d in range(1, 32)
            "time": times,
            "data_format": "netcdf",
            # "area": [47, 6, 36, 19]
        }

        ds = download_cds_dataset(
            "reanalysis-cerra-single-levels",
            request_depth
        )

        datasets.append(ds)

        print(f"Downloaded {year}-{month}")

    # Unione dei mesi
    ds_year = xr.concat(datasets, dim="valid_time")

    # Salvataggio NetCDF
    ds_year.to_netcdf(output_file)

    print(f"Year {year} saved in {output_file}")

    del datasets
    del ds_year

print("\nDownload completed!")
